In [2]:
import pandas as pd

df = pd.read_csv("data/processed/cbe_clean.csv")

#df = df.rename(columns={" ": "review_id"})

In [3]:
print(df.columns)


Index(['review', 'rating', 'date', 'bank'], dtype='str')


In [4]:
print(type(df))        # must be DataFrame
print(df.columns)
print(df.head())

<class 'pandas.DataFrame'>
Index(['review', 'rating', 'date', 'bank'], dtype='str')
             review  rating        date                         bank
0       Good to use       5  2026-05-12  Commercial Bank of Ethiopia
1               cbe       1  2026-05-12  Commercial Bank of Ethiopia
2  https..//digital       1  2026-05-11  Commercial Bank of Ethiopia
3               Cbe       4  2026-05-11  Commercial Bank of Ethiopia
4  best and secured       5  2026-05-11  Commercial Bank of Ethiopia


In [9]:
print("before removing duplicates:", len(df))

before removing duplicates: 1790


In [14]:
import pandas as pd
import hashlib

df = pd.read_csv("data/processed/cbe_clean.csv")

# create review_id from text
df["review_id"] = df["review"].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

# remove duplicates using ID
df = df.drop_duplicates(subset=["review_id"])

print("After removing duplicates:", len(df))

After removing duplicates: 1790


In [15]:
import pandas as pd
import hashlib

df = pd.read_csv("data/processed/boa_clean.csv")

# create review_id from text
df["review_id"] = df["review"].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

# remove duplicates using ID
df = df.drop_duplicates(subset=["review_id"])

print("After removing duplicates:", len(df))

After removing duplicates: 1183


In [11]:
import pandas as pd

df = pd.read_csv("data/processed/boa_clean.csv")

df = df.drop_duplicates(subset=["review"])

print("after removing duplicates:", len(df))

after removing duplicates: 1183


In [12]:
import pandas as pd

df = pd.read_csv("data/processed/dashen_clean.csv")

#df = df.drop_duplicates(subset=["review"])

print("before removing duplicates:", len(df))

before removing duplicates: 843


In [13]:
import pandas as pd

df = pd.read_csv("data/processed/dashen_clean.csv")

df = df.drop_duplicates(subset=["review"])

print("after removing duplicates:", len(df))

after removing duplicates: 843


In [16]:
import pandas as pd
import hashlib

df = pd.read_csv("data/processed/cbe_clean.csv")

# -------------------------
# 1. Document initial state
# -------------------------
print("Initial rows:", len(df))

missing_review = df["review"].isna().sum()
missing_rating = df["rating"].isna().sum()

print("Missing review text:", missing_review)
print("Missing rating:", missing_rating)

# -------------------------
# 2. Drop missing values
# -------------------------
df = df.dropna(subset=["review", "rating"])

# -------------------------
# 3. Document after cleaning
# -------------------------
print("After dropping missing values:", len(df))
print("Dropped rows:", missing_review + missing_rating)

# -------------------------
# 4. Create review_id (optional but recommended)
# -------------------------
df["review_id"] = df["review"].astype(str).apply(
    lambda x: hashlib.md5(x.strip().encode("utf-8")).hexdigest()
)

# -------------------------
# 5. Remove duplicates
# -------------------------
before_dup = len(df)
df = df.drop_duplicates(subset=["review_id"])

print("After removing duplicates:", len(df))
print("Duplicates removed:", before_dup - len(df))

Initial rows: 1790
Missing review text: 0
Missing rating: 0
After dropping missing values: 1790
Dropped rows: 0
After removing duplicates: 1790
Duplicates removed: 0


In [17]:
print(df[["date"]].head())

         date
0  2026-05-12
1  2026-05-12
2  2026-05-11
3  2026-05-11
4  2026-05-11


In [1]:
import pandas as pd
from transformers import pipeline

# Load dataset
df = pd.read_csv("data/processed/cbe_clean.csv")

# Optional: test on smaller sample first
df = df.head(100)

# Load sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Model loaded successfully.")

# Convert reviews to list
reviews = df["review"].astype(str).tolist()

# Run batch prediction
results = sentiment_model(
    reviews,
    batch_size=8,
    truncation=True
)

# Store outputs
sentiments = []
confidences = []

for result in results:
    label = result["label"]
    score = result["score"]

    # Add neutral class manually
    if score < 0.60:
        sentiment = "neutral"
    else:
        sentiment = "positive" if label == "POSITIVE" else "negative"

    sentiments.append(sentiment)
    confidences.append(score)

# Add to dataframe
df["sentiment"] = sentiments
df["confidence"] = confidences

# Preview
print(df[["review", "sentiment", "confidence"]].head())

# Save results
df.to_csv("data/processed/cbe_sentiment.csv", index=False)

print("Sentiment analysis completed successfully.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

C:\Users\Soret\fintech-review-analytics\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Soret\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Model loaded successfully.
             review sentiment  confidence
0       Good to use  positive    0.999846
1               cbe  positive    0.996601
2  https..//digital  positive    0.610911
3               Cbe  positive    0.996601
4  best and secured  positive    0.999819
Sentiment analysis completed successfully.


In [2]:
import pandas as pd
from transformers import pipeline

# Load dataset
df = pd.read_csv("data/processed/boa_clean.csv")

# Optional: test on smaller sample first
df = df.head(100)

# Load sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Model loaded successfully.")

# Convert reviews to list
reviews = df["review"].astype(str).tolist()

# Run batch prediction
results = sentiment_model(
    reviews,
    batch_size=8,
    truncation=True
)

# Store outputs
sentiments = []
confidences = []

for result in results:
    label = result["label"]
    score = result["score"]

    # Add neutral class manually
    if score < 0.60:
        sentiment = "neutral"
    else:
        sentiment = "positive" if label == "POSITIVE" else "negative"

    sentiments.append(sentiment)
    confidences.append(score)

# Add to dataframe
df["sentiment"] = sentiments
df["confidence"] = confidences

# Preview
print(df[["review", "sentiment", "confidence"]].head())

# Save results
df.to_csv("data/processed/boa_clean.csv", index=False)

print("Sentiment analysis completed successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
                                              review sentiment  confidence
0                                 it's very good app  positive    0.999873
1  this app is good but the speed of app is very ...  negative    0.995230
2                                               good  positive    0.999816
3                                       boa the best  positive    0.999843
4         bank of absiniya is best bank in ethiopian  positive    0.999774
Sentiment analysis completed successfully.


In [3]:
import pandas as pd
from transformers import pipeline

# Load dataset
df = pd.read_csv("data/processed/dashen_clean.csv")

# Optional: test on smaller sample first
df = df.head(100)

# Load sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Model loaded successfully.")

# Convert reviews to list
reviews = df["review"].astype(str).tolist()

# Run batch prediction
results = sentiment_model(
    reviews,
    batch_size=8,
    truncation=True
)

# Store outputs
sentiments = []
confidences = []

for result in results:
    label = result["label"]
    score = result["score"]

    # Add neutral class manually
    if score < 0.60:
        sentiment = "neutral"
    else:
        sentiment = "positive" if label == "POSITIVE" else "negative"

    sentiments.append(sentiment)
    confidences.append(score)

# Add to dataframe
df["sentiment"] = sentiments
df["confidence"] = confidences

# Preview
print(df[["review", "sentiment", "confidence"]].head())

# Save results
df.to_csv("data/processed/dashen_clean.csv", index=False)

print("Sentiment analysis completed successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
                                              review sentiment  confidence
0  Very bad customer service line. they won't pic...  negative    0.999816
1                                          smart app  positive    0.999789
2           The application booting time is so bad..  negative    0.999809
3                                               good  positive    0.999816
4                                           excelent  positive    0.999867
Sentiment analysis completed successfully.


In [4]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv("data/processed/cbe_clean.csv")

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# -------------------------
# Sentiment function
# -------------------------
def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))

    compound = score["compound"]

    if compound >= 0.05:
        sentiment = "positive"
    elif compound <= -0.05:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return pd.Series([sentiment, compound])

# Apply analysis
df[["vader_sentiment", "vader_score"]] = df["review"].apply(vader_sentiment)

# Preview
print(df[["review", "vader_sentiment", "vader_score"]].head())

# Save
df.to_csv("data/processed/cbe_vader_sentiment.csv", index=False)

print("VADER sentiment completed.")

             review vader_sentiment  vader_score
0       Good to use        positive       0.4404
1               cbe         neutral       0.0000
2  https..//digital         neutral       0.0000
3               Cbe         neutral       0.0000
4  best and secured        positive       0.7845
VADER sentiment completed.


In [5]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv("data/processed/dashen_clean.csv")

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# -------------------------
# Sentiment function
# -------------------------
def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))

    compound = score["compound"]

    if compound >= 0.05:
        sentiment = "positive"
    elif compound <= -0.05:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return pd.Series([sentiment, compound])

# Apply analysis
df[["vader_sentiment", "vader_score"]] = df["review"].apply(vader_sentiment)

# Preview
print(df[["review", "vader_sentiment", "vader_score"]].head())

# Save
df.to_csv("data/processed/cbe_vader_sentiment.csv", index=False)

print("VADER sentiment completed.")

                                              review vader_sentiment  \
0  Very bad customer service line. they won't pic...        negative   
1                                          smart app        positive   
2           The application booting time is so bad..        negative   
3                                               good        positive   
4                                           excelent         neutral   

   vader_score  
0      -0.8357  
1       0.4019  
2      -0.6696  
3       0.4404  
4       0.0000  
VADER sentiment completed.


In [6]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load dataset
df = pd.read_csv("data/processed/boa_clean.csv")

# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# -------------------------
# Sentiment function
# -------------------------
def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))

    compound = score["compound"]

    if compound >= 0.05:
        sentiment = "positive"
    elif compound <= -0.05:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return pd.Series([sentiment, compound])

# Apply analysis
df[["vader_sentiment", "vader_score"]] = df["review"].apply(vader_sentiment)

# Preview
print(df[["review", "vader_sentiment", "vader_score"]].head())

# Save
df.to_csv("data/processed/cbe_vader_sentiment.csv", index=False)

print("VADER sentiment completed.")

                                              review vader_sentiment  \
0                                 it's very good app        positive   
1  this app is good but the speed of app is very ...        positive   
2                                               good        positive   
3                                       boa the best        positive   
4         bank of absiniya is best bank in ethiopian        positive   

   vader_score  
0       0.4927  
1       0.2382  
2       0.4404  
3       0.6369  
4       0.6369  
VADER sentiment completed.


In [7]:
import pandas as pd

# Load sentiment dataset
df = pd.read_csv("data/processed/cbe_sentiment.csv")

# -------------------------
# 1. Aggregate by bank
# -------------------------
bank_sentiment = df.groupby("bank")["confidence"].mean()

print("\nAverage Sentiment Confidence by Bank")
print(bank_sentiment)

# -------------------------
# 2. Aggregate by star rating
# -------------------------
rating_sentiment = df.groupby("rating")["confidence"].mean()

print("\nAverage Sentiment Confidence by Rating")
print(rating_sentiment)

# -------------------------
# 3. Count sentiments per bank
# -------------------------
bank_sentiment_counts = df.groupby("bank")["sentiment"].value_counts()

print("\nSentiment Counts by Bank")
print(bank_sentiment_counts)

# -------------------------
# 4. Mean sentiment score by bank and rating
# -------------------------
bank_rating_sentiment = df.groupby(
    ["bank", "rating"]
)["confidence"].mean()

print("\nMean Sentiment Confidence by Bank and Rating")
print(bank_rating_sentiment)

# -------------------------
# 5. Save results
# -------------------------
bank_sentiment.to_csv(
    "data/processed/bank_sentiment_summary.csv"
)

rating_sentiment.to_csv(
    "data/processed/rating_sentiment_summary.csv"
)

print("\nAggregation completed successfully.")


Average Sentiment Confidence by Bank
bank
Commercial Bank of Ethiopia    0.978587
Name: confidence, dtype: float64

Average Sentiment Confidence by Rating
rating
1    0.964434
2    0.999802
3    0.991634
4    0.988079
5    0.980117
Name: confidence, dtype: float64

Sentiment Counts by Bank
bank                         sentiment
Commercial Bank of Ethiopia  positive     60
                             negative     40
Name: count, dtype: int64

Mean Sentiment Confidence by Bank and Rating
bank                         rating
Commercial Bank of Ethiopia  1         0.964434
                             2         0.999802
                             3         0.991634
                             4         0.988079
                             5         0.980117
Name: confidence, dtype: float64

Aggregation completed successfully.


In [8]:
import pandas as pd

# Load sentiment dataset
df = pd.read_csv("data/processed/dashen_clean.csv")

# -------------------------
# 1. Aggregate by bank
# -------------------------
bank_sentiment = df.groupby("bank")["confidence"].mean()

print("\nAverage Sentiment Confidence by Bank")
print(bank_sentiment)

# -------------------------
# 2. Aggregate by star rating
# -------------------------
rating_sentiment = df.groupby("rating")["confidence"].mean()

print("\nAverage Sentiment Confidence by Rating")
print(rating_sentiment)

# -------------------------
# 3. Count sentiments per bank
# -------------------------
bank_sentiment_counts = df.groupby("bank")["sentiment"].value_counts()

print("\nSentiment Counts by Bank")
print(bank_sentiment_counts)

# -------------------------
# 4. Mean sentiment score by bank and rating
# -------------------------
bank_rating_sentiment = df.groupby(
    ["bank", "rating"]
)["confidence"].mean()

print("\nMean Sentiment Confidence by Bank and Rating")
print(bank_rating_sentiment)

# -------------------------
# 5. Save results
# -------------------------
bank_sentiment.to_csv(
    "data/processed/bank_sentiment_summary.csv"
)

rating_sentiment.to_csv(
    "data/processed/rating_sentiment_summary.csv"
)

print("\nAggregation completed successfully.")


Average Sentiment Confidence by Bank
bank
Dashen Bank    0.95756
Name: confidence, dtype: float64

Average Sentiment Confidence by Rating
rating
1    0.953223
2    0.999159
3    0.998916
4    0.982607
5    0.952294
Name: confidence, dtype: float64

Sentiment Counts by Bank
bank         sentiment
Dashen Bank  positive     61
             negative     38
             neutral       1
Name: count, dtype: int64

Mean Sentiment Confidence by Bank and Rating
bank         rating
Dashen Bank  1         0.953223
             2         0.999159
             3         0.998916
             4         0.982607
             5         0.952294
Name: confidence, dtype: float64

Aggregation completed successfully.


In [9]:
import pandas as pd

# Load sentiment dataset
df = pd.read_csv("data/processed/boa_clean.csv")

# -------------------------
# 1. Aggregate by bank
# -------------------------
bank_sentiment = df.groupby("bank")["confidence"].mean()

print("\nAverage Sentiment Confidence by Bank")
print(bank_sentiment)

# -------------------------
# 2. Aggregate by star rating
# -------------------------
rating_sentiment = df.groupby("rating")["confidence"].mean()

print("\nAverage Sentiment Confidence by Rating")
print(rating_sentiment)

# -------------------------
# 3. Count sentiments per bank
# -------------------------
bank_sentiment_counts = df.groupby("bank")["sentiment"].value_counts()

print("\nSentiment Counts by Bank")
print(bank_sentiment_counts)

# -------------------------
# 4. Mean sentiment score by bank and rating
# -------------------------
bank_rating_sentiment = df.groupby(
    ["bank", "rating"]
)["confidence"].mean()

print("\nMean Sentiment Confidence by Bank and Rating")
print(bank_rating_sentiment)

# -------------------------
# 5. Save results
# -------------------------
bank_sentiment.to_csv(
    "data/processed/bank_sentiment_summary.csv"
)

rating_sentiment.to_csv(
    "data/processed/rating_sentiment_summary.csv"
)

print("\nAggregation completed successfully.")


Average Sentiment Confidence by Bank
bank
Bank of Abyssinia    0.969906
Name: confidence, dtype: float64

Average Sentiment Confidence by Rating
rating
1    0.989674
2    0.998359
3    0.907222
4    0.958153
5    0.966576
Name: confidence, dtype: float64

Sentiment Counts by Bank
bank               sentiment
Bank of Abyssinia  positive     60
                   negative     40
Name: count, dtype: int64

Mean Sentiment Confidence by Bank and Rating
bank               rating
Bank of Abyssinia  1         0.989674
                   2         0.998359
                   3         0.907222
                   4         0.958153
                   5         0.966576
Name: confidence, dtype: float64

Aggregation completed successfully.


In [10]:
import pandas as pd

df = pd.read_csv("data/processed/cbe_sentiment.csv")

# -------------------------
# Theme classification function
# -------------------------
def assign_theme(review):
    review = str(review).lower()

    if any(word in review for word in [
        "login", "password", "otp", "signin", "access"
    ]):
        return "Account Access Issues"

    elif any(word in review for word in [
        "transfer", "transaction", "payment", "delay"
    ]):
        return "Transaction Performance"

    elif any(word in review for word in [
        "crash", "bug", "freeze", "error"
    ]):
        return "App Stability"

    elif any(word in review for word in [
        "design", "ui", "interface", "layout"
    ]):
        return "UI & Design"

    elif any(word in review for word in [
        "support", "service", "help"
    ]):
        return "Customer Support"

    elif any(word in review for word in [
        "feature", "update", "add"
    ]):
        return "Feature Requests"

    else:
        return "Other"

# Apply themes
df["theme"] = df["review"].apply(assign_theme)

# Preview
print(df[["review", "theme"]].head())

# Theme counts
print("\nTheme Distribution:")
print(df["theme"].value_counts())

# Save
df.to_csv("data/processed/cbe_themes.csv", index=False)

print("Theme extraction completed.")

             review  theme
0       Good to use  Other
1               cbe  Other
2  https..//digital  Other
3               Cbe  Other
4  best and secured  Other

Theme Distribution:
theme
Other                      79
Feature Requests            7
Transaction Performance     5
Customer Support            5
App Stability               2
UI & Design                 1
Account Access Issues       1
Name: count, dtype: int64
Theme extraction completed.


In [11]:
import pandas as pd

df = pd.read_csv("data/processed/dashen_clean.csv")

# -------------------------
# Theme classification function
# -------------------------
def assign_theme(review):
    review = str(review).lower()

    if any(word in review for word in [
        "login", "password", "otp", "signin", "access"
    ]):
        return "Account Access Issues"

    elif any(word in review for word in [
        "transfer", "transaction", "payment", "delay"
    ]):
        return "Transaction Performance"

    elif any(word in review for word in [
        "crash", "bug", "freeze", "error"
    ]):
        return "App Stability"

    elif any(word in review for word in [
        "design", "ui", "interface", "layout"
    ]):
        return "UI & Design"

    elif any(word in review for word in [
        "support", "service", "help"
    ]):
        return "Customer Support"

    elif any(word in review for word in [
        "feature", "update", "add"
    ]):
        return "Feature Requests"

    else:
        return "Other"

# Apply themes
df["theme"] = df["review"].apply(assign_theme)

# Preview
print(df[["review", "theme"]].head())

# Theme counts
print("\nTheme Distribution:")
print(df["theme"].value_counts())

# Save
df.to_csv("data/processed/dashen_clean.csv", index=False)

print("Theme extraction completed.")

                                              review             theme
0  Very bad customer service line. they won't pic...  Customer Support
1                                          smart app             Other
2           The application booting time is so bad..             Other
3                                               good             Other
4                                           excelent             Other

Theme Distribution:
theme
Other                      86
Feature Requests            3
UI & Design                 3
Transaction Performance     3
Customer Support            2
Account Access Issues       2
App Stability               1
Name: count, dtype: int64
Theme extraction completed.


In [12]:
import pandas as pd

df = pd.read_csv("data/processed/boa_clean.csv")

# -------------------------
# Theme classification function
# -------------------------
def assign_theme(review):
    review = str(review).lower()

    if any(word in review for word in [
        "login", "password", "otp", "signin", "access"
    ]):
        return "Account Access Issues"

    elif any(word in review for word in [
        "transfer", "transaction", "payment", "delay"
    ]):
        return "Transaction Performance"

    elif any(word in review for word in [
        "crash", "bug", "freeze", "error"
    ]):
        return "App Stability"

    elif any(word in review for word in [
        "design", "ui", "interface", "layout"
    ]):
        return "UI & Design"

    elif any(word in review for word in [
        "support", "service", "help"
    ]):
        return "Customer Support"

    elif any(word in review for word in [
        "feature", "update", "add"
    ]):
        return "Feature Requests"

    else:
        return "Other"

# Apply themes
df["theme"] = df["review"].apply(assign_theme)

# Preview
print(df[["review", "theme"]].head())

# Theme counts
print("\nTheme Distribution:")
print(df["theme"].value_counts())

# Save
df.to_csv("data/processed/boa_clean.csv", index=False)

print("Theme extraction completed.")

                                              review  theme
0                                 it's very good app  Other
1  this app is good but the speed of app is very ...  Other
2                                               good  Other
3                                       boa the best  Other
4         bank of absiniya is best bank in ethiopian  Other

Theme Distribution:
theme
Other                      81
Transaction Performance     6
Customer Support            5
Account Access Issues       4
App Stability               4
Name: count, dtype: int64
Theme extraction completed.


In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv("data/processed/cbe_clean.csv")

# Remove missing reviews
df = df.dropna(subset=["review"])

# Convert to string
reviews = df["review"].astype(str)

# -------------------------
# TF-IDF Vectorizer
# -------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=50
)

# Fit and transform
X = vectorizer.fit_transform(reviews)

# Get feature names
features = vectorizer.get_feature_names_out()

# Calculate mean TF-IDF score
scores = X.mean(axis=0).A1

# Create dataframe
keywords_df = pd.DataFrame({
    "keyword": features,
    "tfidf_score": scores
})

# Sort descending
keywords_df = keywords_df.sort_values(
    by="tfidf_score",
    ascending=False
)

# Display results
print(keywords_df.head(20))

# Save results
keywords_df.to_csv(
    "data/processed/cbe_keywords.csv",
    index=False
)

print("Keyword extraction completed.")

        keyword  tfidf_score
3           app     0.138659
20         good     0.090462
9          best     0.049384
11          cbe     0.039385
7          bank     0.038203
42       update     0.037247
30         nice     0.035646
43          use     0.031392
4   application     0.030817
23         like     0.029035
46         work     0.025164
14         easy     0.022886
8       banking     0.020642
38         time     0.020443
47      working     0.020009
34      service     0.019552
18         fast     0.019423
49           ነው     0.018741
21     good app     0.018031
17    excellent     0.017706
Keyword extraction completed.


In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv("data/processed/boa_clean.csv")

# Remove missing reviews
df = df.dropna(subset=["review"])

# Convert to string
reviews = df["review"].astype(str)

# -------------------------
# TF-IDF Vectorizer
# -------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=50
)

# Fit and transform
X = vectorizer.fit_transform(reviews)

# Get feature names
features = vectorizer.get_feature_names_out()

# Calculate mean TF-IDF score
scores = X.mean(axis=0).A1

# Create dataframe
keywords_df = pd.DataFrame({
    "keyword": features,
    "tfidf_score": scores
})

# Sort descending
keywords_df = keywords_df.sort_values(
    by="tfidf_score",
    ascending=False
)

# Display results
print(keywords_df.head(20))

# Save results
keywords_df.to_csv(
    "data/processed/boa_clean.csv",
    index=False
)

print("Keyword extraction completed.")

     keyword  tfidf_score
2        app     0.143427
16      good     0.125675
6       best     0.060612
7        boa     0.057732
4       bank     0.056425
23      nice     0.045312
37      slow     0.037780
14      fast     0.035407
32   service     0.034918
17  good app     0.030134
24  nice app     0.021193
46      work     0.021094
49     ጥሩ ነው     0.020000
45       use     0.019513
26        ok     0.019053
48     worst     0.018737
13  ethiopia     0.017400
12     error     0.017163
47   working     0.016823
22     money     0.016597
Keyword extraction completed.


In [16]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv("data/processed/dashen_clean.csv")

# Remove missing reviews
df = df.dropna(subset=["review"])

# Convert to string
reviews = df["review"].astype(str)

# -------------------------
# TF-IDF Vectorizer
# -------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=50
)

# Fit and transform
X = vectorizer.fit_transform(reviews)

# Get feature names
features = vectorizer.get_feature_names_out()

# Calculate mean TF-IDF score
scores = X.mean(axis=0).A1

# Create dataframe
keywords_df = pd.DataFrame({
    "keyword": features,
    "tfidf_score": scores
})

# Sort descending
keywords_df = keywords_df.sort_values(
    by="tfidf_score",
    ascending=False
)

# Display results
print(keywords_df.head(20))

# Save results
keywords_df.to_csv(
    "data/processed/dashen_clean.csv",
    index=False
)

print("Keyword extraction completed.")

           keyword  tfidf_score
1              app     0.165443
15            good     0.092591
4             bank     0.059155
7             best     0.058487
24            nice     0.055036
10          dashen     0.047538
36           super     0.041436
13            fast     0.027933
16           great     0.027901
42             use     0.027237
37       super app     0.026780
40          update     0.026624
20            like     0.025016
3             apps     0.024877
22          mobile     0.023925
8         best app     0.022262
46         working     0.021216
2      application     0.021166
12            easy     0.020747
23  mobile banking     0.020395
Keyword extraction completed.


In [20]:
import pandas as pd

# Load your processed dataset (change path if needed)
df = pd.read_csv("data/processed/cbe_themes.csv")

# -------------------------
# Standardize column names
# -------------------------
df = df.rename(columns={
    "review": "review_text",
    "sentiment": "sentiment_label",
    "confidence": "sentiment_score",
    "theme": "identified_theme"
})

# -------------------------
# Ensure review_id exists
# (create if missing)
# -------------------------
if "review_id" not in df.columns:
    import hashlib

    df["review_id"] = df["review_text"].astype(str).apply(
        lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
    )

# -------------------------
# Select required columns only
# -------------------------
final_df = df[
    ["review_id", "review_text", "sentiment_label", "sentiment_score", "identified_theme"]
]

# -------------------------
# Save final dataset
# -------------------------
output_path = "data/processed/cbe_final_dataset.csv"
final_df.to_csv(output_path, index=False)

print(f"Final dataset saved → {output_path}")
print(final_df.head())

Final dataset saved → data/processed/cbe_final_dataset.csv
                          review_id       review_text sentiment_label  \
0  8046b1ddc9a1fbcd32fdc1255c700eb1       Good to use        positive   
1  ff42374122f85b5ecf60c1ad31c10df2               cbe        positive   
2  95e002a459cb9eea14a95bfee4ad752c  https..//digital        positive   
3  682aaddf08ed9b04e35bc54ad2897e89               Cbe        positive   
4  93474d858abed39bd5a6d9aab135e478  best and secured        positive   

   sentiment_score identified_theme  
0         0.999846            Other  
1         0.996601            Other  
2         0.610911            Other  
3         0.996601            Other  
4         0.999819            Other  


In [ ]:
import pandas as pd

# Load your processed dataset (change path if needed)
df = pd.read_csv("data/processed/cbe_themes.csv")

# -------------------------
# Standardize column names
# -------------------------
df = df.rename(columns={
    "review": "review_text",
    "sentiment": "sentiment_label",
    "confidence": "sentiment_score",
    "theme": "identified_theme"
})

# -------------------------
# Ensure review_id exists
# (create if missing)
# -------------------------
if "review_id" not in df.columns:
    import hashlib

    df["review_id"] = df["review_text"].astype(str).apply(
        lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
    )

# -------------------------
# Select required columns only
# -------------------------
final_df = df[
    ["review_id", "review_text", "sentiment_label", "sentiment_score", "identified_theme"]
]

# -------------------------
# Save final dataset
# -------------------------
output_path = "data/processed/cbe_final_dataset.csv"
final_df.to_csv(output_path, index=False)

print(f"Final dataset saved → {output_path}")
print(final_df.head())